# Search API Overview — every flavor of `/search`

> Reference notebook for the Visual Search retrieval surface. Read this
> **before** the agent notebooks — every agent that retrieves uses one
> or more of these patterns.

## What `/search` can do

| Variant | When to use |
|---|---|
| Semantic by clip (`BY_CLIP`) | "find moments where X is happening" — natural language → video segments |
| Semantic by audio (`BY_AUDIO`) | "find moments where someone says X" — semantic match against transcripts |
| Exact-phrase transcript (`/search_audio_transcripts`) | "find the literal phrase 'turn left'" — LIKE matching |
| By tag | "all videos tagged sop-audit" — boolean filter on user-defined tags set at upload time |
| By camera_tag | "all videos shot on DRIVETHRU-CAM-A" — boolean filter on the camera_model property |
| Time-windowed | combine any of the above with `datetime_taken` for date-range queries |
| Image similarity | `/search_similar_images`, `/search_clips_by_image` — see Visual Search docs |

All variants share the same response shape:
`[{videoNo, videoName, startTime, endTime, score, audio_ts?}, ...]`.


## Setup

You need:

1. A Memories.ai API key (`sk-mavi-...`) — get one at the [Memories.ai console](https://api-platform.memories.ai/stripe).
2. Python 3.10+ and the `requests` library (`pip install requests`).

Set the key as an environment variable before launching Jupyter, or paste it
inline in the cell below. The same key works across Visual Search, Visual
Intelligence, and Visual Agents — no separate auth per product.


In [ ]:
import os, json, time, requests

# ────────────────────────────────────────────────────────────────────────────
# Auth: the Memories.ai key is a single token used across every product.
# Pass it as the literal `Authorization` header value — no `Bearer` prefix.
# ────────────────────────────────────────────────────────────────────────────
API_KEY = os.environ.get("MEMORIES_API_KEY") or "sk-mavi-..."  # ← paste here if not using env
HEADERS = {"Authorization": API_KEY}

# Hosts: Visual Search and Visual Intelligence live on different domains.
VS_HOST  = "https://api.memories.ai/serve/api/v1"            # Visual Search
VLM_HOST = "https://mavi-backend.memories.ai/serve/api/v2"   # Visual Intelligence (VLM + Visual Agents)

# Default VLM model used for verification / reasoning. Other options include
# `qwen:qwen2.5-vl-72b-instruct`, `nova:amazon.nova-lite-v1:0`, etc. — see
# Memories.ai docs for the full list and per-token pricing.
VLM_MODEL = "gemini:gemini-2.5-flash"


In [ ]:
def search(query, *, video_nos=None, unique_id="default", top_k=10,
           filtering_level="medium", search_type="BY_CLIP",
           datetime_taken=None, tag=None, camera_tag=None,
           latitude=None, longitude=None, max_retries=3):
    """Visual Search — POST /search.

    Returns the list of {videoNo, startTime, endTime, score, ...} matches.

    Filter parameters (all optional, combinable):
      • video_nos       restrict to specific videos (up to 100 ids)
      • datetime_taken  filter to videos captured at-or-after this timestamp
      • tag             filter to videos carrying this user-defined tag
      • camera_tag      filter to videos shot on this camera_model
      • latitude/longitude  GPS proximity filter (must be paired)

    Notes on the envelope: Visual Search responses are wrapped in
    {code, msg, data, success, failed}. code="0000" means OK; the actual
    hit list is in `data`. code="0001" ("network abnormal") is transient
    and we retry it.
    """
    body = {
        "search_param": query,
        "search_type": search_type,                 # "BY_CLIP" or "BY_AUDIO"
        "unique_id": unique_id,                      # namespace within your account
        "top_k": top_k,
        "filtering_level": filtering_level,          # "low"|"medium"|"high"
    }
    if video_nos:        body["video_nos"]     = list(video_nos)
    if datetime_taken:   body["datetime_taken"] = datetime_taken
    if tag:              body["tag"]            = tag
    if camera_tag:       body["camera_tag"]     = camera_tag
    if latitude is not None and longitude is not None:
        body["latitude"]  = latitude
        body["longitude"] = longitude

    for attempt in range(max_retries):
        r = requests.post(f"{VS_HOST}/search", headers=HEADERS, json=body, timeout=60)
        r.raise_for_status()
        envelope = r.json()
        code = envelope.get("code")
        if code == "0000":
            return envelope.get("data") or []
        if code == "0001" and attempt < max_retries - 1:
            # Transient. Backoff and retry.
            time.sleep(0.4 * (2 ** attempt))
            continue
        raise RuntimeError(f"/search failed: code={code} msg={envelope.get('msg')!r}")
    return []


In [ ]:
def search_transcripts(query, *, unique_id="default", page=1, page_size=100):
    """Visual Search — GET /search_audio_transcripts.

    Exact-phrase LIKE matching over stored audio transcripts. Use when you
    know roughly what was said. For semantic audio search, use /search with
    search_type="BY_AUDIO" instead.
    """
    r = requests.get(
        f"{VS_HOST}/search_audio_transcripts",
        headers=HEADERS,
        params={"query": query, "unique_id": unique_id, "page": page, "page_size": page_size},
        timeout=30,
    )
    r.raise_for_status()
    envelope = r.json()
    assert envelope.get("code") == "0000", envelope
    return envelope.get("data") or {}


## 1 — Semantic search by clip (`BY_CLIP`)

The default. Returns *video segments* whose visual content matches the
query by meaning, not keyword. Internally the index stores VLM-generated
captions + embeddings; the query is embedded and matched against those.


In [ ]:
hits = search(
    "a person walking outdoors",
    top_k=5,
    filtering_level="medium",   # discard hits below score=0.225
)
for h in hits:
    print(f"  {h['videoNo']}  {h['startTime']}-{h['endTime']}s  score={h['score']:.3f}")


## 2 — Semantic search by audio (`BY_AUDIO`)

Same endpoint, `search_type="BY_AUDIO"`. Matches against the audio
transcript channel — finds moments where someone *says* something
like the query. The returned hits include an `audio_ts` field with
the matched transcript snippet.


In [ ]:
hits = search(
    "talking about cooking pasta",
    search_type="BY_AUDIO",
    top_k=5,
    filtering_level="low",
)
for h in hits:
    print(f"  {h['videoNo']}  t={h['startTime']}s  audio: {h.get('audio_ts', '')[:80]!r}")


## 3 — Exact-phrase transcript search

Different endpoint: `GET /search_audio_transcripts`. Use when you know
the literal phrase that was said. LIKE matching, not semantic — case-
insensitive substring match.


In [ ]:
tx = search_transcripts("I'll have", page_size=10)
videos = tx.get("videos", [])
print(f"{len(videos)} transcript matches\n")
for v in videos[:5]:
    print(f"  {v['videoNo']}  t={v['startTime']}s  audio: {v.get('audio_ts', '')[:80]!r}")


## 4 — Search by tag

Tags are user-defined strings you set at upload time (`tags=["sop-audit",
"drivethru"]`). The `/search` endpoint can filter to videos carrying a
specific tag. Combine with a semantic query to narrow ("compliance
moments **in sop-audit footage**") or pass an empty query to enumerate
("everything tagged X").

Set tags at upload like so:

```python
# During upload (Visual Search /upload):
requests.post(f"{VS_HOST}/upload", headers=HEADERS,
              files={"file": (name, f, "video/mp4")},
              data={"unique_id": "default",
                    "tags": ["sop-audit", "drivethru"]})
```

Then search filtered to that tag:


In [ ]:
# Filter semantic search to videos tagged 'sop-audit'.
hits = search(
    "staff handing a bag to a customer",
    tag="sop-audit",
    top_k=5,
    filtering_level="low",
)
print(f"{len(hits)} hits within tag='sop-audit'")
for h in hits[:5]:
    print(f"  {h['videoNo']}  {h['startTime']}-{h['endTime']}s  score={h['score']:.3f}")


## 5 — Search by camera_tag (camera_model filter)

Same idea, but filters on the `camera_model` property set at upload
time (e.g. `camera_model="DRIVETHRU-CAM-A"`). Useful when you want
to look only at a specific camera or vehicle.


In [ ]:
# Restrict to a specific camera. Pair with a semantic query for best results.
hits = search(
    "vehicle parking in the bay",
    camera_tag="DRIVETHRU-CAM-A",
    top_k=5,
    filtering_level="low",
)
print(f"{len(hits)} hits from camera DRIVETHRU-CAM-A")


## 6 — Date-window filtering with `datetime_taken`

Filter to videos captured at-or-after a timestamp. Useful for
time-series queries ("everything from last Tuesday's shift").

Note: the API filters to *at-or-after* only; to bound the upper end,
do a client-side filter on the response.


In [ ]:
# "What was on camera after 11am on 2026-05-05?"
hits = search(
    "a person",
    datetime_taken="2026-05-05 11:00:00",
    top_k=5,
    filtering_level=None,    # no score threshold here
)
print(f"{len(hits)} hits captured at or after 2026-05-05 11:00:00")


## 7 — Combining filters

Every filter is composable. Most agents use *some* combination — e.g.
SOP Compliance uses `tag="sop-audit"` + `video_nos=[shift_vid]` +
a semantic query.


In [ ]:
# Compound filter: SOP-audit footage + specific camera + recent + a semantic query.
hits = search(
    "staff at the pickup window",
    tag="sop-audit",
    camera_tag="DRIVETHRU-CAM-A",
    datetime_taken="2026-05-01 00:00:00",
    top_k=10,
    filtering_level="medium",
)
print(f"{len(hits)} composite-filter hits")


## 8 — Image-based retrieval *(see Visual Search docs)*

Two image-based endpoints round out the surface; they're not used by
the agents in this collection so we don't run them here, but the
signatures are:

- `POST /search_similar_images` — library-wide visual similarity to a
  query image (`multipart/form-data`).
- `POST /search_clips_by_image` — find time-ranges within a specific
  video that match a query image, optionally with a refining prompt.

See [Visual Search → Search Your Library](https://github.com/Memories-ai-labs/api-docs/blob/main/visual-search/search-private-video.mdx)
for the full request/response shapes.

---

## Where to go next

- Pick the agent notebook closest to your use case (01-08).
- Each agent uses one or two of the patterns above. The filter
  arguments are the same `search()` signature you used here.
